# dip-fmm parameter sweep

Compare regular MagTense with cached, non-periodic spherical dip-fmm plans. The regular reference is run once for each grid; dip-fmm is then swept over orders 1–10 and depths 2–5. The core prints initialization and evaluation time separately.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

repository_root = Path.cwd()
while repository_root != repository_root.parent and not (repository_root / 'python' / 'src').is_dir():
    repository_root = repository_root.parent
example_dir = repository_root / 'python' / 'examples' / 'micromagnetism' / 'FMM'
if str(example_dir) not in sys.path:
    sys.path.insert(0, str(example_dir))

from fmm_vs_regular import run_sweep

In [ ]:
GRID_SIZES = (15, 20, 25, 30)
ORDERS = range(1, 11)
DEPTHS = (2, 3, 4, 5)
USE_CUDA = True  # Requires MagTense to have been built with USE_CUDA=1.

In [ ]:
rows = run_sweep(
    grid_sizes=GRID_SIZES,
    orders=ORDERS,
    depths=DEPTHS,
    use_cuda=USE_CUDA,
)
rows[:5]

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
for ax, grid_size in zip(axes.flat, GRID_SIZES):
    for depth in DEPTHS:
        curve = [row for row in rows if row['grid_size'] == grid_size and row['depth'] == depth]
        x = [row['order'] for row in curve]
        y = np.maximum([row['relative_rms'] for row in curve], np.finfo(float).tiny)
        ax.semilogy(x, y, marker='o', label=f'depth {depth}')
    ax.set_title(f'{grid_size}³ particles')
    ax.grid(alpha=0.3)
    ax.legend()
for ax in axes[-1, :]:
    ax.set_xlabel('Expansion order')
for ax in axes[:, 0]:
    ax.set_ylabel('Final-state relative RMS')
fig.tight_layout()
plt.show()